# Integrating four levels of protein structure into one function prediction

**Protein Function Predictor: the integration step**

The four analysis modules each describe a protein at one structural level:

| Level | What the module does | What it tells us about function |
|---|---|---|
| Primary | DIAMOND search of Swiss-Prot, HMMER search of Pfam | *which* characterised protein or family the query is related to, and how closely |
| Secondary | JPred + ProtBert prediction, compared with experimental PDB structures | whether the query's local structure matches solved structures of its relatives |
| Tertiary | SWISS-MODEL 3D model, Foldseek structural search | whether the query's fold matches known structures (Foldseek gives a homology probability) |
| Quaternary | UniProt, Complex Portal, RCSB PDB, PDBe PISA, SWISS-MODEL, IntAct, AlphaFold DB | the oligomeric state and complexes of the related protein |

This notebook turns those four results into **one predicted function with a confidence score**,
and explains every step of how it got there.

**The idea in one line:** function is predicted by *transferring* the annotation of a related,
characterised protein, so the confidence is

$$P(\text{function correct}) = P(\text{query is homologous to the template}) \times P(\text{function is conserved at this evolutionary distance}),$$

after which the structural levels act as **consistency checks** that raise or lower this estimate.

All data in this notebook are real outputs of the four modules (run through the project's web app);
no tool is re-run here, so the notebook runs in seconds and needs no internet.

## 1. Setup
Only `numpy`, `pandas` and `matplotlib` are needed (pre-installed on Kaggle and Colab).

In [ ]:
import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 2. Why not simply add up (or average) the four scores?

A weighted sum treats the four levels as four independent opinions about function. Biologically they are not:

1. **The levels are not independent.** The tertiary model is built by SWISS-MODEL from a template found by
   *sequence* search; the secondary-structure comparison uses PDB entries found by *sequence* search; the
   quaternary pipeline studies the *sequence* hit itself. Adding their scores counts the same homologue several times.
2. **They answer different questions.** Sequence and structure answer *"is the query related to a known protein?"*.
   Function transfer additionally needs *"if it is related, is the function still the same at this distance?"*.
   Enzymes illustrate this: 40% identity is enough to transfer the first three EC digits confidently, but all four
   digits need more than 60% identity for 90% accuracy (Tian & Skolnick 2003), and function starts to diverge
   below about 70% identity (Rost 2002).
3. **The E-value is a weak guide to function.** Tian & Skolnick found the magnitude of the PSI-BLAST E-value to be only
   weakly correlated with conservation of enzyme function, so this model uses *percent identity* for the
   conservation term and keeps E-values only as significance filters.
4. **Averaging over "whatever ran" inflates weak evidence.** If only one level produced a result, a renormalised
   average gives that one result the full weight (shown concretely in Section 9).

## 3. The model: a chain of evidence

### Step 1: choose the route (what is transferred, and how specific it is)
The first route that applies is used:

| Route | Condition | Predicted function | P(homology) | P(function conserved) |
|---|---|---|---|---|
| **1. Sequence** | DIAMOND hit passed the primary thresholds (identity > 30%, coverage ≥ 80%) and is a characterised protein | the hit's protein name (+ its GO molecular functions) | 0.99 if identity ≥ 40%; 0.90 if 30–40% | 0.95 if ≥ 60%; 0.80 if 40–60%; 0.55 if 30–40% |
| **2. Domain** | a Pfam domain passed (independent E-value < 1e-10) | "*X* family member" | 0.95 | 0.50 |
| **3. Fold** | a significant Foldseek match (E ≤ 1e-3, prob ≥ 0.5) of a reliable model (mean QMEAN ≥ 0.6) | "Structural similarity to *X*" | Foldseek's own `prob` | 0.25 |
| none | — | "Function not determined" | — | — |

* The identity cut-offs come from the twilight-zone analysis of sequence alignments (Rost 1999: below ~30–40%
  identity, homology becomes uncertain) and the enzyme-function curves of Tian & Skolnick (2003).
* Foldseek's `prob` is, by its documentation, the *"estimated probability for query and target to be homologous
  (e.g. being within the same SCOPe superfamily)"*, so it is used directly as P(homology).
* A shared fold without other evidence of common ancestry is rarely associated with a shared interaction
  (Aloy et al. 2003), and similar folds often carry different functions, so the fold route stays low.
* Every prediction states its **specificity**: specific protein, protein family, or fold only.

### Step 2: consistency checks (Bayesian updating on the log-odds scale)
Starting from $\text{logit}(P_\text{base})$, each other level adds a log-likelihood-ratio-like adjustment:

| Check | Agrees | Contradicts |
|---|---|---|
| Tertiary: reliable model (mean QMEAN ≥ 0.6) with a significant Foldseek match | +0.5 | no significant match: −1.0 |
| Secondary: agreement with experimental PDB structures (≥ 20 residues) | ≥ 75%: +0.4 | < 60%: −1.0 |
| Quaternary: the template's assembly evidence | CONFIRMED / SUPPORTED: +0.4 | CONFLICTING: −0.5 |

* **Agreement is rewarded less than contradiction is penalised, on purpose.** If the query really is homologous
  to the template, agreement is *expected* (and these checks partly reuse the same template), so it carries
  little new information; disagreement is surprising and therefore informative.
* The secondary thresholds follow from predictor accuracy: three-state predictors are right for roughly 80% of
  residues, so a shared fold should give about 75% agreement or more.
* Close homologues (≥ 30–40% identity) almost invariably interact in the same way (Aloy et al. 2003); below 40%
  identity the quaternary adjustment is therefore halved.
* An unreliable 3D model (low QMEAN) is not used as evidence either way.

Final score $= \sigma\big(\text{logit}(P_\text{base}) + \sum \Delta\big)$. Bands: **high ≥ 0.80**, **moderate ≥ 0.50**, low below.
The score is a literature-anchored estimate, *not* a calibrated probability (see Section 12).

## 4. Reading the four modules' outputs
The functions below read the results exactly as the web app stores them (one `LevelResult` per level) and keep
only the evidence the model needs. They are also used in Section 5b to load new proteins downloaded from the app.

In [ ]:
import math
import re

GENERIC_GO = {
    "protein binding", "identical protein binding", "enzyme binding",
    "protein homodimerization activity", "protein heterodimerization activity", "molecular_function",
}
UNCHARACTERISED = re.compile(r"^(putative\s+)?(uncharacteri[sz]ed|UPF\d+)\b", re.IGNORECASE)


def _num(value):
    """A finite number, or None."""
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) else None


def _live(level):
    return isinstance(level, dict) and level.get("source") == "live"


def _passed(level, prefix):
    """The primary notebook's own verdict: its evidence line is positive when the test passed."""
    return any(item.get("polarity") == "positive" and str(item.get("text", "")).startswith(prefix)
               for item in level.get("evidence", []))


def _parse_swissprot_title(title):
    """'sp|P69905|HBA_HUMAN Hemoglobin subunit alpha OS=Homo sapiens OX=9606 ...' -> parts."""
    token, _, rest = title.partition(" ")
    parts = token.split("|")
    accession = parts[1] if len(parts) > 2 else token
    entry = parts[2] if len(parts) > 2 else token
    protein = rest.split(" OS=")[0].strip()
    organism = re.search(r"OS=(.*?)(?: OX=|$)", rest)
    return accession, entry, protein, organism.group(1).strip() if organism else ""


def primary_evidence(level):
    """DIAMOND best Swiss-Prot hit and the best Pfam domain (HMMER) from the primary level."""
    if not _live(level):
        return {"ran": False}
    raw = level.get("raw") or {}
    diamond, hmmer = raw.get("diamond") or {}, raw.get("hmmer") or {}
    hit = None
    if diamond.get("title") and (_num(diamond.get("identity")) or 0) > 0:
        accession, entry, protein, organism = _parse_swissprot_title(str(diamond["title"]))
        hit = {
            "accession": accession, "entry": entry, "protein": protein, "organism": organism,
            "identity": _num(diamond.get("identity")), "coverage": _num(diamond.get("query_coverage")),
            "evalue": _num(diamond.get("e_value")),
            "passed": _passed(level, "DIAMOND:"),  # identity > 30% and coverage >= 80%
        }
    domain = None
    if _passed(level, "HMMER:") and hmmer.get("domain") not in (None, "None"):
        accession = str(hmmer.get("accession") or "")
        same = [d for d in hmmer.get("domains", []) or [] if d.get("accession") == accession and _num(d.get("e_value")) is not None]
        best = min(same, key=lambda d: float(d["e_value"])) if same else {}
        domain = {
            "name": str(hmmer["domain"]), "accession": accession.split(".")[0],
            "description": str(best.get("description") or hmmer["domain"]), "evalue": _num(hmmer.get("e_value")),
        }
    return {"ran": True, "hit": hit, "domain": domain}


def secondary_evidence(level):
    """Best agreement between the predicted secondary structure and experimental PDB structures."""
    if not _live(level):
        return {"ran": False}
    raw = level.get("raw") or {}
    rows = [
        r for r in raw.get("experimental_comparison") or []
        if _num(r.get("SS_Agreement_%")) is not None and _num(r.get("Comparable_Residues")) is not None
    ]
    best = max(rows, key=lambda r: (float(r["Comparable_Residues"]) >= 20, float(r["SS_Agreement_%"]))) if rows else None
    return {
        "ran": True,
        "predictor_agreement": _num(raw.get("predictor_agreement_percent")),
        "best": None if best is None else {
            "entity": str(best.get("PDB_entity")), "identity": _num(best.get("Sequence_Identity_%")),
            "residues": int(float(best["Comparable_Residues"])), "agreement": float(best["SS_Agreement_%"]),
        },
    }


def _pretty_target(database, token):
    alphafold = re.match(r"^AF-([A-Za-z0-9]+)-F\d+", token)
    pdb = re.match(r"^([0-9][A-Za-z0-9]{3})(?:[-_.]|$)", token)
    if alphafold:
        return f"AlphaFold DB {alphafold.group(1)}"
    if pdb and database.lower().startswith("pdb"):
        chain = token.rsplit("_", 1)[1] if "_" in token else ""
        return f"PDB {pdb.group(1).upper()}" + (f" chain {chain}" if chain else "")
    return token


def tertiary_evidence(level):
    """Model quality (mean local QMEAN) and the best Foldseek match of the SWISS-MODEL model."""
    if not _live(level):
        return {"ran": False}
    raw = level.get("raw") or {}
    qmean = next((_num(m.get("value")) for m in level.get("metrics", []) if m.get("label") == "Mean local quality"), None)
    best = None
    for database, hit in (raw.get("foldseek_top_hits") or {}).items():
        evalue = _num((hit or {}).get("eval"))
        if evalue is None:
            continue
        token, _, description = str(hit.get("target", "")).partition(" ")
        candidate = {
            "database": database, "target": _pretty_target(database, token), "description": description.strip(),
            "evalue": evalue, "prob": _num(hit.get("prob")), "seq_id": _num(hit.get("seqId")),
        }
        if best is None or evalue < best["evalue"]:
            best = candidate
    return {"ran": True, "qmean": qmean, "best": best}


def quaternary_evidence(level):
    """What the quaternary pipeline found for the analysed protein (assembly, complexes, GO terms)."""
    if not _live(level):
        return {"ran": False}
    raw = level.get("raw") or {}
    output = raw.get("pipeline_output") or {}
    assembly = output.get("target_self_assembly") or {}
    go_terms = []
    for term in (raw.get("uniprot") or {}).get("go_terms", []) or []:
        aspect, _, name = str(term.get("name", "")).partition(":")
        if aspect == "F" and name and name.lower() not in GENERIC_GO:
            go_terms.append([str(term.get("id")), name])
    go_terms.sort(key=lambda t: "activity" not in t[1].lower())  # activities first
    return {
        "ran": True,
        "mode": raw.get("input"),  # "homologue" or "sequence"
        "analysed": raw.get("analysed_target"),
        "status": str(assembly.get("confidence_status") or output.get("status") or "UNKNOWN"),
        "formula": assembly.get("formula"),
        "assembly_type": assembly.get("assembly_type"),
        "subunits": assembly.get("subunit_count"),
        "complexes": [[c.get("name"), c.get("complex_portal_id"), c.get("stoichiometry")]
                      for c in output.get("functional_complexes") or []],
        "go_mf": go_terms,
    }


def evidence_from_levels(levels):
    """levels: {'primary': LevelResult dict or None, 'secondary': ..., 'tertiary': ..., 'quaternary': ...}"""
    return {
        "primary": primary_evidence(levels.get("primary")),
        "secondary": secondary_evidence(levels.get("secondary")),
        "tertiary": tertiary_evidence(levels.get("tertiary")),
        "quaternary": quaternary_evidence(levels.get("quaternary")),
    }

## 5. The data: real outputs of the four modules

Eight proteins analysed with the project's web app (results cached by the app, November 2026). They were chosen
to cover every route of the model, and some deliberately lack levels, to show how missing evidence is handled:

| Protein | Levels available | Why it is included |
|---|---|---|
| Hemoglobin α | all four | well-characterised protein, oligomeric (α2β2) |
| Ubiquitin | all four | well-characterised monomer |
| SNRPG-like peptide B | primary, secondary | close homologue (78%), plus a secondary-structure check |
| SNRPG-like peptide A | primary | the same family without the secondary check (compare with B) |
| AIG1-like protein | primary | homology test failed (37%, coverage 59%) but a Pfam domain passed |
| Unknown sequence | primary, secondary, tertiary | no relatives anywhere: should stay undetermined |
| AKT1, Cytochrome c | tertiary only | strong structural matches but no sequence analysis |

The quaternary results are those of the team's quaternary pipeline: hemoglobin α (P69905: homotetramer A4,
CONFIRMED by PDBe PISA on PDB 1A01, Complex Portal CPX-2158) and ubiquitin (closest Swiss-Prot entry Q865C5:
monomer, SUPPORTED).

In [ ]:
EVIDENCE = json.loads(r'''
[
 {
  "name": "Hemoglobin α",
  "note": "app example, UniProt P69905",
  "length": 142,
  "primary": {
   "ran": true,
   "hit": {
    "accession": "P69905",
    "entry": "HBA_HUMAN",
    "protein": "Hemoglobin subunit alpha",
    "organism": "Homo sapiens",
    "identity": 100.0,
    "coverage": 100.0,
    "evalue": 2.94e-100,
    "passed": true
   },
   "domain": {
    "name": "Globin",
    "accession": "PF00042",
    "description": "Globin",
    "evalue": 2.9e-33
   }
  },
  "secondary": {
   "ran": true,
   "predictor_agreement": 88.02816901408451,
   "best": {
    "entity": "1BZ1_1",
    "identity": 100.0,
    "residues": 125,
    "agreement": 96.8
   }
  },
  "tertiary": {
   "ran": true,
   "qmean": 0.98,
   "best": {
    "database": "pdb100",
    "target": "PDB 6KAI chain E",
    "description": "Crosslinked alpha(Ni)-beta(Fe) human hemoglobin A in the T quaternary structure at 95 K: Light",
    "evalue": 1.284e-19,
    "prob": 1.0,
    "seq_id": 100.0
   }
  },
  "quaternary": {
   "ran": true,
   "mode": "homologue",
   "analysed": "P69905",
   "status": "CONFIRMED",
   "formula": "A4",
   "assembly_type": "HOMOMER",
   "subunits": 4,
   "complexes": [
    [
     "Hemoglobin HbA complex",
     "CPX-2158",
     "A2B2"
    ]
   ],
   "go_mf": [
    [
     "GO:0005344",
     "oxygen carrier activity"
    ],
    [
     "GO:0020037",
     "heme binding"
    ],
    [
     "GO:0005506",
     "iron ion binding"
    ],
    [
     "GO:0019825",
     "oxygen binding"
    ]
   ]
  }
 },
 {
  "name": "Ubiquitin",
  "note": "app example, PDB 1UBQ",
  "length": 76,
  "primary": {
   "ran": true,
   "hit": {
    "accession": "Q865C5",
    "entry": "UBIQ_CAMDR",
    "protein": "Ubiquitin",
    "organism": "Camelus dromedarius",
    "identity": 100.0,
    "coverage": 100.0,
    "evalue": 2.36e-48,
    "passed": true
   },
   "domain": {
    "name": "ubiquitin",
    "accession": "PF00240",
    "description": "Ubiquitin family",
    "evalue": 7.7e-34
   }
  },
  "secondary": {
   "ran": true,
   "predictor_agreement": 75.0,
   "best": {
    "entity": "1FXT_2",
    "identity": 100.0,
    "residues": 57,
    "agreement": 98.24561403508771
   }
  },
  "tertiary": {
   "ran": true,
   "qmean": 0.96,
   "best": {
    "database": "afdb50",
    "target": "AlphaFold DB A0A8D0HA87",
    "description": "Ubiquitin-like domain-containing protein",
    "evalue": 4.909e-15,
    "prob": 1.0,
    "seq_id": 100.0
   }
  },
  "quaternary": {
   "ran": true,
   "mode": "homologue",
   "analysed": "Q865C5",
   "status": "SUPPORTED",
   "formula": "A",
   "assembly_type": "MONOMER",
   "subunits": 1,
   "complexes": [],
   "go_mf": []
  }
 },
 {
  "name": "SNRPG-like peptide B",
  "note": "test sequence, 53 aa",
  "length": 53,
  "primary": {
   "ran": true,
   "hit": {
    "accession": "Q3ZBL0",
    "entry": "RUXG_BOVIN",
    "protein": "Small nuclear ribonucleoprotein G",
    "organism": "Bos taurus",
    "identity": 78.0,
    "coverage": 94.33962264150944,
    "evalue": 7.48e-21,
    "passed": true
   },
   "domain": {
    "name": "LSM",
    "accession": "PF01423",
    "description": "LSM domain",
    "evalue": 1e-11
   }
  },
  "secondary": {
   "ran": true,
   "predictor_agreement": 75.47169811320755,
   "best": {
    "entity": "4V98_8",
    "identity": 75.47169811320755,
    "residues": 40,
    "agreement": 92.5
   }
  },
  "tertiary": {
   "ran": false
  },
  "quaternary": {
   "ran": false
  }
 },
 {
  "name": "SNRPG-like peptide A",
  "note": "test sequence, 50 aa",
  "length": 50,
  "primary": {
   "ran": true,
   "hit": {
    "accession": "Q3ZBL0",
    "entry": "RUXG_BOVIN",
    "protein": "Small nuclear ribonucleoprotein G",
    "organism": "Bos taurus",
    "identity": 78.0,
    "coverage": 100.0,
    "evalue": 9.35e-21,
    "passed": true
   },
   "domain": {
    "name": "LSM",
    "accession": "PF01423",
    "description": "LSM domain",
    "evalue": 9.7e-12
   }
  },
  "secondary": {
   "ran": false
  },
  "tertiary": {
   "ran": false
  },
  "quaternary": {
   "ran": false
  }
 },
 {
  "name": "AIG1-like protein",
  "note": "test sequence, 184 aa",
  "length": 184,
  "primary": {
   "ran": true,
   "hit": {
    "accession": "Q9D8B1",
    "entry": "AIG1_MOUSE",
    "protein": "Androgen-induced gene 1 protein",
    "organism": "Mus musculus",
    "identity": 37.1,
    "coverage": 59.23913043478261,
    "evalue": 4.98e-19,
    "passed": false
   },
   "domain": {
    "name": "Far-17a_AIG1",
    "accession": "PF04750",
    "description": "FAR-17a/AIG1-like protein",
    "evalue": 2.1e-31
   }
  },
  "secondary": {
   "ran": false
  },
  "tertiary": {
   "ran": false
  },
  "quaternary": {
   "ran": false
  }
 },
 {
  "name": "Unknown sequence",
  "note": "test sequence with no known relatives, 55 aa",
  "length": 55,
  "primary": {
   "ran": true,
   "hit": null,
   "domain": null
  },
  "secondary": {
   "ran": true,
   "predictor_agreement": 74.54545454545455,
   "best": null
  },
  "tertiary": {
   "ran": true,
   "qmean": 0.34,
   "best": {
    "database": "pdb100",
    "target": "PDB 6WES chain A",
    "description": "Crystal structure of the effector SnTox3 from Parastagonospora nodorum",
    "evalue": 3.088,
    "prob": 0.855,
    "seq_id": 24.0
   }
  },
  "quaternary": {
   "ran": false
  }
 },
 {
  "name": "AKT1",
  "note": "app example, UniProt P31749",
  "length": 480,
  "primary": {
   "ran": false
  },
  "secondary": {
   "ran": false
  },
  "tertiary": {
   "ran": true,
   "qmean": 0.83,
   "best": {
    "database": "afdb50",
    "target": "AlphaFold DB A0A3Q2HHJ1",
    "description": "non-specific serine/threonine protein kinase",
    "evalue": 1.312e-79,
    "prob": 1.0,
    "seq_id": 97.7
   }
  },
  "quaternary": {
   "ran": false
  }
 },
 {
  "name": "Cytochrome c",
  "note": "test sequence, 105 aa",
  "length": 105,
  "primary": {
   "ran": false
  },
  "secondary": {
   "ran": false
  },
  "tertiary": {
   "ran": true,
   "qmean": 0.98,
   "best": {
    "database": "pdb100",
    "target": "PDB 3ZOO chain A",
    "description": "Structure of the Y46F mutant of human cytochrome c",
    "evalue": 9.9e-21,
    "prob": 1.0,
    "seq_id": 99.0
   }
  },
  "quaternary": {
   "ran": false
  }
 }
]
''')
print(f"{len(EVIDENCE)} proteins loaded:", ", ".join(p['name'] for p in EVIDENCE))

### 5b. Optional: add your own proteins
In the web app, open a finished analysis and click **Download JSON**. Add the downloaded `.json` files to this
notebook (Kaggle: *Add Input* → upload them as a dataset; Colab: upload them into a folder called `analyses`).
The cell below finds them automatically; with no files it does nothing.

In [ ]:
def evidence_from_app_download(path):
    job = json.loads(Path(path).read_text(encoding="utf-8"))
    levels = {level["key"]: (level.get("result") if level.get("status") == "completed" else None) for level in job["levels"]}
    return {"name": job["query"]["name"], "note": f"downloaded from the app ({Path(path).name})",
            "length": job["query"]["length"], **evidence_from_levels(levels)}

added = []
for folder in (Path("/kaggle/input"), Path("analyses")):
    if folder.is_dir():
        for path in sorted(folder.rglob("*.json")):
            try:
                added.append(evidence_from_app_download(path))
            except (KeyError, TypeError, ValueError, AttributeError) as error:
                print(f"Skipped {path.name}: not an analysis downloaded from the app ({error.__class__.__name__})")
EVIDENCE.extend(added)
print(f"Added {len(added)} protein(s); {len(EVIDENCE)} in total.")

### What each level found

In [ ]:
def overview_row(p):
    prim, sec, ter, quat = p["primary"], p["secondary"], p["tertiary"], p["quaternary"]
    hit = prim.get("hit") if prim["ran"] else None
    dom = prim.get("domain") if prim["ran"] else None
    best_ss = sec.get("best") if sec["ran"] else None
    best_fs = ter.get("best") if ter["ran"] else None
    return {
        "Protein": p["name"],
        "Swiss-Prot hit (identity / coverage)": ("not run" if not prim["ran"] else "none" if hit is None else
            f"{hit['entry']} ({hit['identity']:.1f}% / {hit['coverage']:.1f}%){'' if hit['passed'] else ' - failed'}"),
        "Pfam domain": "not run" if not prim["ran"] else (f"{dom['name']} (E={dom['evalue']:.1e})" if dom else "none"),
        "SS agreement with PDB": "not run" if not sec["ran"] else (f"{best_ss['agreement']:.1f}% ({best_ss['entity']})" if best_ss else "no structure"),
        "Model QMEAN": "not run" if not ter["ran"] else ter.get("qmean"),
        "Foldseek best (E / prob)": "not run" if not ter["ran"] else (
            f"{best_fs['target']} ({best_fs['evalue']:.1e} / {best_fs['prob']})" if best_fs else "none"),
        "Quaternary": "not run" if not quat["ran"] else f"{quat['analysed']}: {quat['formula']} {quat['status']}",
    }

display(pd.DataFrame([overview_row(p) for p in EVIDENCE]).set_index("Protein"))

## 6. The model in code
All numbers live in one dictionary, `PARAMS`, so they can be inspected and varied (Section 10).

In [ ]:
PARAMS = {
    # Route 1 - sequence homology (DIAMOND hit that passed the primary thresholds)
    "seq_hom_40": 0.99,       # P(homologous) when identity >= 40%
    "seq_hom_30": 0.90,       # P(homologous) when identity is 30-40% (twilight zone, Rost 1999)
    "cons_60": 0.95,          # P(function conserved) at >= 60% identity (Tian & Skolnick 2003)
    "cons_40": 0.80,          # ... at 40-60% identity (first three EC digits reliable)
    "cons_30": 0.55,          # ... at 30-40% identity (function diverges, Rost 2002)
    # Route 2 - Pfam domain (HMMER independent E-value < 1e-10)
    "dom_hom": 0.95,          # P(the domain is a true family member)
    "dom_cons": 0.50,         # P(the family's function applies to this protein)
    # Route 3 - fold only (Foldseek)
    "fold_cons": 0.25,        # P(function conserved | same fold only) (Aloy et al. 2003)
    "fold_evalue": 1e-3,      # a structural match counts if E <= this ...
    "fold_prob": 0.5,         # ... and Foldseek's homology probability >= this
    "qmean_min": 0.6,         # the 3D model is trusted only if mean local QMEAN >= this
    # Consistency checks (log-odds adjustments)
    "d_ter_agree": 0.5,       # reliable model with a significant structural match
    "d_ter_contra": -1.0,     # reliable model but no significant structural match
    "ss_agree": 75.0,         # % agreement with experimental structures that counts as agreement
    "ss_contra": 60.0,        # % agreement below which it counts as a contradiction
    "ss_min_res": 20,         # minimum residues compared
    "d_ss_agree": 0.4,
    "d_ss_contra": -1.0,
    "d_q_good": 0.4,          # template's assembly CONFIRMED or SUPPORTED
    "d_q_conflict": -0.5,     # template's assembly evidence CONFLICTING
    "q_full_identity": 40.0,  # below this identity the quaternary adjustment is halved (Aloy et al. 2003)
    # Confidence bands
    "high": 0.80,
    "moderate": 0.50,
}


def logit(p):
    p = min(max(p, 1e-3), 1 - 1e-3)
    return math.log(p / (1 - p))


def sigmoid(x):
    return 1 / (1 + math.exp(-x))


def p_homology_seq(identity, P):
    return P["seq_hom_40"] if identity >= 40 else P["seq_hom_30"]


def p_conserved_seq(identity, P):
    if identity >= 60:
        return P["cons_60"]
    if identity >= 40:
        return P["cons_40"]
    return P["cons_30"]


def fold_is_significant(tertiary, P):
    best = tertiary.get("best") if tertiary.get("ran") else None
    return bool(best) and best["evalue"] <= P["fold_evalue"] and (best["prob"] is None or best["prob"] >= P["fold_prob"])


def model_is_reliable(tertiary, P):
    return tertiary.get("ran") and tertiary.get("qmean") is not None and tertiary["qmean"] >= P["qmean_min"]


def family_label(description):
    lowered = description.lower()
    if lowered.endswith("family"):
        return f"{description} member"
    if lowered.endswith("domain"):
        return f"{description}-containing protein"
    return f"{description} family"


def predict(ev, P=PARAMS):
    """Chain of evidence: P(homology) x P(function conserved), then consistency checks in log-odds."""
    prim, sec, ter, quat = ev["primary"], ev["secondary"], ev["tertiary"], ev["quaternary"]
    hit = prim.get("hit") if prim.get("ran") else None
    domain = prim.get("domain") if prim.get("ran") else None
    named = hit is not None and hit["passed"] and not UNCHARACTERISED.match(hit["protein"] or hit["entry"])

    # ---- Step 1: choose the route (what is transferred, and how specific it is)
    if named:
        route, specificity = "sequence", "specific protein"
        function = hit["protein"]
        p_hom, p_cons = p_homology_seq(hit["identity"], P), p_conserved_seq(hit["identity"], P)
        basis = (f"closest Swiss-Prot protein {hit['accession']} ({hit['entry']}), "
                 f"{hit['identity']:.1f}% identity over {hit['coverage']:.1f}% of the query")
    elif domain is not None:
        route, specificity = "domain", "protein family"
        function = f"{family_label(domain['description'])} (Pfam {domain['accession']})"
        p_hom, p_cons = P["dom_hom"], P["dom_cons"]
        basis = f"Pfam domain {domain['description']} ({domain['accession']}, E = {domain['evalue']:.1e})"
    elif fold_is_significant(ter, P) and model_is_reliable(ter, P):
        best = ter["best"]
        route, specificity = "fold", "fold only"
        function = f"Structural similarity to {best['target']}" + (f" ({best['description']})" if best["description"] else "")
        p_hom, p_cons = (best["prob"] if best["prob"] is not None else 0.5), P["fold_cons"]
        basis = f"Foldseek match {best['target']} in {best['database']} (E = {best['evalue']:.1e}, prob = {p_hom:.2f})"
    else:
        route, specificity, function, p_hom, p_cons, basis = None, "none", "Function not determined", None, None, ""

    support = {key: ("not run" if not ev[key].get("ran") else "neutral") for key in ("primary", "secondary", "tertiary", "quaternary")}
    steps = []
    if route is None:
        return {"function": function, "route": None, "specificity": specificity, "score": None,
                "band": "undetermined", "steps": steps, "support": support, "basis": basis,
                "p_hom": None, "p_cons": None, "base": None}

    base = p_hom * p_cons
    total = logit(base)
    steps.append(("base", f"P(homology) {p_hom:.2f} x P(function conserved) {p_cons:.2f} = {base:.2f}", total))
    if route in ("sequence", "domain"):
        support["primary"] = "supports"
    else:
        support["tertiary"] = "supports"

    # ---- Step 2: consistency checks
    if route != "fold" and ter.get("ran"):
        if not model_is_reliable(ter, P):
            steps.append(("tertiary", f"model unreliable (mean QMEAN {ter.get('qmean')}); not used", 0.0))
        elif fold_is_significant(ter, P):
            b = ter["best"]
            steps.append(("tertiary", f"reliable model matches {b['target']} (E = {b['evalue']:.1e})", P["d_ter_agree"]))
            support["tertiary"] = "supports"
        else:
            steps.append(("tertiary", "reliable model but no significant structural match", P["d_ter_contra"]))
            support["tertiary"] = "contradicts"

    if sec.get("ran"):
        best = sec.get("best")
        if best is None or best["residues"] < P["ss_min_res"]:
            steps.append(("secondary", "no experimental structure to compare with", 0.0))
        elif best["agreement"] >= P["ss_agree"]:
            steps.append(("secondary", f"{best['agreement']:.1f}% agreement with {best['entity']} ({best['residues']} residues)", P["d_ss_agree"]))
            support["secondary"] = "supports"
        elif best["agreement"] < P["ss_contra"]:
            steps.append(("secondary", f"only {best['agreement']:.1f}% agreement with {best['entity']}", P["d_ss_contra"]))
            support["secondary"] = "contradicts"
        else:
            steps.append(("secondary", f"{best['agreement']:.1f}% agreement with {best['entity']} (inconclusive)", 0.0))

    if quat.get("ran"):
        about_template = route == "sequence" and quat.get("mode") == "homologue" and quat.get("analysed") == hit["accession"]
        if not about_template:
            steps.append(("quaternary", "the pipeline did not analyse the predicted template", 0.0))
        else:
            factor = 1.0 if hit["identity"] >= P["q_full_identity"] else 0.5
            status = quat["status"]
            if status in ("CONFIRMED", "SUPPORTED"):
                steps.append(("quaternary", f"template assembly {quat['formula']} is {status}", P["d_q_good"] * factor))
                support["quaternary"] = "supports"
            elif status == "CONFLICTING":
                steps.append(("quaternary", "template assembly evidence is CONFLICTING", P["d_q_conflict"] * factor))
                support["quaternary"] = "contradicts"
            else:
                steps.append(("quaternary", f"template assembly status {status} (no adjustment)", 0.0))

    total = sum(delta for _, _, delta in steps)
    score = sigmoid(total)
    band = "high" if score >= P["high"] else "moderate" if score >= P["moderate"] else "low"
    return {"function": function, "route": route, "specificity": specificity, "score": score, "band": band,
            "steps": steps, "support": support, "basis": basis, "p_hom": p_hom, "p_cons": p_cons, "base": base}

## 7. The base confidence as a function of sequence identity
For the sequence route, the starting confidence is P(homology) × P(function conserved). The steps sit at the
literature thresholds: 30% (primary test), 40% (three EC digits, Tian & Skolnick) and 60% (all four EC digits).

In [ ]:
identities = np.linspace(30.01, 100, 600)
base_curve = [p_homology_seq(i, PARAMS) * p_conserved_seq(i, PARAMS) for i in identities]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(identities, base_curve, color="#2348c8", lw=2.5, label="sequence route: P(homology) x P(conserved)")
ax.axhline(PARAMS["dom_hom"] * PARAMS["dom_cons"], color="#e08a1e", ls="--", label="domain route (family level)")
ax.axhline(PARAMS["fold_cons"], color="#8a8f98", ls=":", label="fold route, upper bound (prob = 1)")
for x, text in ((40, "40%: 3 EC digits"), (60, "60%: 4 EC digits")):
    ax.axvline(x, color="#cccccc", lw=1)
    ax.text(x + 0.8, 0.12, text, fontsize=8, color="#555555")
points = {}
for p in EVIDENCE:
    hit = p["primary"].get("hit") if p["primary"]["ran"] else None
    if hit and hit["passed"]:
        points.setdefault(round(hit["identity"], 1), []).append(p["name"])
for identity, protein_names in points.items():
    y = p_homology_seq(identity, PARAMS) * p_conserved_seq(identity, PARAMS)
    ax.scatter(identity, y, color="black", zorder=3)
    ax.annotate(", ".join(protein_names), (identity, y), textcoords="offset points", xytext=(-6, -14), fontsize=8, ha="right")
ax.set(xlabel="sequence identity to the closest Swiss-Prot protein (%)", ylabel="base confidence", ylim=(0, 1.02), xlim=(30, 101))
ax.legend(loc="lower right", fontsize=8, frameon=False)
ax.set_title("Starting confidence before the consistency checks")
plt.tight_layout()
plt.show()

## 8. Predictions

In [ ]:
RESULTS = [(p, predict(p)) for p in EVIDENCE]

def summary_row(p, r):
    return {
        "Protein": p["name"],
        "Predicted function": r["function"],
        "Specificity": r["specificity"],
        "Base": "" if r["base"] is None else f"{r['base']:.2f}",
        "Confidence": "" if r["score"] is None else f"{r['score']:.2f}",
        "Band": r["band"],
        **{k.capitalize(): v for k, v in r["support"].items()},
    }

display(pd.DataFrame([summary_row(p, r) for p, r in RESULTS]).set_index("Protein"))

### Step-by-step explanation for each protein
This is the same reasoning the web app will show under *"How this verdict was reached"*.

In [ ]:
def explain(p, r):
    lines = [f"### {p['name']} · {p['note']}", f"**Prediction:** {r['function']}  ·  specificity: {r['specificity']}"]
    if r["route"] is None:
        lines.append("No route applied: no Swiss-Prot homologue or Pfam domain passed the primary tests, and no "
                     "significant structural match came from a reliable model. **Function not determined.**")
    else:
        lines.append(f"**Route {['sequence', 'domain', 'fold'].index(r['route']) + 1} ({r['route']}):** {r['basis']}.")
        lines.append("")
        lines.append("| Step | Evidence | Log-odds |")
        lines.append("|---|---|---|")
        for level, text, delta in r["steps"]:
            shown = f"{delta:+.2f}" if level != "base" else f"{delta:.2f} (start)"
            lines.append(f"| {level} | {text} | {shown} |")
        total = sum(d for _, _, d in r["steps"])
        lines.append(f"| **total** | σ({total:.2f}) | **{r['score']:.2f} → {r['band']}** |")
    if r["route"] == "sequence":
        quat = p["quaternary"]
        if quat.get("ran") and quat.get("analysed") == p["primary"]["hit"]["accession"]:
            context = []
            if quat.get("formula"):
                context.append(f"oligomeric state of the template: {quat['formula']} ({quat['assembly_type'].lower()}, {quat['status']})")
            context += [f"member of the {name} ({cpx}, {stoich})" for name, cpx, stoich in quat["complexes"]]
            if quat["go_mf"]:
                context.append("GO molecular function of the template: " + ", ".join(t for _, t in quat["go_mf"][:4]))
            if context:
                lines.append("")
                lines.append("**Biological context from the quaternary level:** " + "; ".join(context) + ".")
    display(Markdown("\n".join(lines)))

for p, r in RESULTS:
    explain(p, r)

### Where each protein's confidence comes from
The bars show the starting log-odds (from P(homology) × P(conserved)) and each check's adjustment;
the dot is the final log-odds, and the number on the right is the final confidence.

In [ ]:
scored = [(p, r) for p, r in RESULTS if r["score"] is not None]
colours = {"base": "#2348c8", "tertiary": "#3fa34d", "secondary": "#e08a1e", "quaternary": "#8e44ad"}
fig, ax = plt.subplots(figsize=(9, 0.6 * len(scored) + 1.2))
for row, (p, r) in enumerate(scored):
    left_pos, left_neg = 0.0, 0.0
    for level, _, delta in r["steps"]:
        if delta == 0:
            continue
        hatch = "//" if delta < 0 and level != "base" else None  # a contradiction
        if delta > 0:
            ax.barh(row, delta, left=left_pos, color=colours[level], edgecolor="white")
            left_pos += delta
        else:
            ax.barh(row, delta, left=left_neg, color=colours[level], edgecolor="white", hatch=hatch)
            left_neg += delta
    total = sum(d for _, _, d in r["steps"])
    ax.plot(total, row, "ko", ms=5)
    ax.text(max(left_pos, total) + 0.15, row, f"{r['score']:.2f} ({r['band']})", va="center", fontsize=8)
for x, label in ((logit(PARAMS["high"]), "high"), (logit(PARAMS["moderate"]), "moderate")):
    ax.axvline(x, color="#999999", ls="--", lw=1)
    ax.text(x, 1.01, label, fontsize=8, color="#666666", ha="center", transform=ax.get_xaxis_transform())
ax.axvline(0, color="black", lw=0.8)
ax.set_yticks(range(len(scored)), [p["name"] for p, _ in scored])
ax.invert_yaxis()
ax.set_xlabel("log-odds (hatched = contradiction; a negative base means a base confidence below 0.5)")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colours.values()]
ax.legend(handles, ["base (homology x conservation)", "tertiary check", "secondary check", "quaternary check"],
          fontsize=8, frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## 9. Comparison with a naive weighted average
For contrast, the "obvious" approach: give each level a weight (sequence 0.40, domain 0.15, fold 0.20,
secondary 0.10, quaternary 0.15), score each from 0 to 1, and average over the levels that produced a result.

In [ ]:
NAIVE_WEIGHTS = {"sequence": 0.40, "domain": 0.15, "fold": 0.20, "secondary": 0.10, "quaternary": 0.15}

def naive_weighted_average(ev):
    prim, sec, ter, quat = ev["primary"], ev["secondary"], ev["tertiary"], ev["quaternary"]
    scores = {}
    if prim["ran"]:
        hit = prim.get("hit")
        scores["sequence"] = 0.0 if not hit or not hit["passed"] else 1.0 if hit["identity"] >= 60 else 0.7 if hit["identity"] >= 40 else 0.4
        scores["domain"] = 1.0 if prim.get("domain") else 0.0
    if ter["ran"]:
        scores["fold"] = 1.0 if fold_is_significant(ter, PARAMS) else 0.0
    if sec["ran"] and sec.get("best") and sec["best"]["residues"] >= 20:
        a = sec["best"]["agreement"]
        scores["secondary"] = 1.0 if a >= 75 else 0.5 if a >= 60 else 0.0
    if quat["ran"] and quat.get("mode") == "homologue":
        scores["quaternary"] = {"CONFIRMED": 1.0, "SUPPORTED": 0.75, "PREDICTED": 0.4}.get(quat["status"], 0.0)
    if not scores:
        return None
    return sum(NAIVE_WEIGHTS[k] * v for k, v in scores.items()) / sum(NAIVE_WEIGHTS[k] for k in scores)

rows = []
for p, r in RESULTS:
    naive = naive_weighted_average(p)
    rows.append({"Protein": p["name"], "Specificity": r["specificity"],
                 "Naive weighted average": "" if naive is None else f"{naive:.2f}",
                 "Chain of evidence": "" if r["score"] is None else f"{r['score']:.2f}", "Band": r["band"]})
display(pd.DataFrame(rows).set_index("Protein"))

**What the comparison shows**
* For well-characterised proteins (hemoglobin α, ubiquitin) both methods agree: everything points the same way.
* With **only a structural match** (AKT1, cytochrome c: tertiary only), the naive average gives the maximum score
  (1.00) because the single available level gets all the weight. Yet a shared fold alone says little about the
  exact function. The chain of evidence keeps these at ≤ 0.25 and labels them *fold only*.
  (Both proteins would be predicted specifically and with high confidence once the primary level runs: their
  structural matches share > 90% sequence identity.)
* For the **AIG1-like protein**, the naive average mixes a failed homology test (0) with a passing domain (1) into an
  uninterpretable 0.27, whereas the chain of evidence gives a *family-level* prediction with a confidence that
  reflects how often a family's function carries over.
* The naive approach has no notion of **specificity**, cannot express that agreement between dependent levels is
  expected, and lets contradictions only dilute rather than count against a prediction.

## 10. Sensitivity analysis: do the conclusions depend on the exact numbers?
Each parameter is changed on its own by −20% and +20% (the E-value threshold by a factor of 10), and every
protein is re-scored. A robust model should keep its bands.

In [ ]:
SKIP = {"high", "moderate"}  # reporting thresholds, not model parameters

def perturbed(key, factor):
    P = dict(PARAMS)
    if key == "fold_evalue":
        P[key] = PARAMS[key] * (10.0 if factor > 1 else 0.1)
    else:
        P[key] = PARAMS[key] * factor
        if key.startswith(("seq_hom", "cons_", "dom_", "fold_cons", "fold_prob")):
            P[key] = min(P[key], 0.999)
    return P

baseline = {p["name"]: r for p, r in RESULTS}
records, ranges = [], {p["name"]: [] for p in EVIDENCE}
for key in PARAMS:
    if key in SKIP:
        continue
    for factor in (0.8, 1.2):
        P = perturbed(key, factor)
        changed = []
        for p in EVIDENCE:
            r = predict(p, P)
            if r["score"] is not None:
                ranges[p["name"]].append(r["score"])
            if r["band"] != baseline[p["name"]]["band"] or r["function"] != baseline[p["name"]]["function"]:
                changed.append(f"{p['name']}: {baseline[p['name']]['band']} → {r['band']}")
        records.append({"Parameter": key, "Change": "x10" if key == "fold_evalue" and factor > 1 else
                        "/10" if key == "fold_evalue" else f"{factor - 1:+.0%}", "Band or function changes": "; ".join(changed) or "none"})

table = pd.DataFrame(records)
stable = (table["Band or function changes"] == "none").sum()
print(f"Bands and functions unchanged in {stable} of {len(table)} single-parameter perturbations.")
display(table[table["Band or function changes"] != "none"].reset_index(drop=True))

**Reading the result.** On the eight proteins above, no perturbation changes any *predicted function*, and the
few band changes all concern proteins supported by a single level of evidence that sit close to a band boundary
(for example, SNRPG-like peptide A, which has only the primary level). Proteins supported by several
consistent levels keep their band under every perturbation, which is exactly the behaviour expected of an
evidence-combination model: confidence that rests on several independent-looking checks is harder to overturn.

In [ ]:
names = [p["name"] for p, r in RESULTS if r["score"] is not None]
fig, ax = plt.subplots(figsize=(8, 0.5 * len(names) + 1))
for row, name in enumerate(names):
    lo, hi = min(ranges[name]), max(ranges[name])
    ax.plot([lo, hi], [row, row], color="#2348c8", lw=6, alpha=0.35, solid_capstyle="round")
    ax.plot(baseline[name]["score"], row, "ko")
ax.axvline(PARAMS["high"], color="#999999", ls="--", lw=1)
ax.axvline(PARAMS["moderate"], color="#999999", ls="--", lw=1)
ax.set_yticks(range(len(names)), names)
ax.invert_yaxis()
ax.set(xlim=(0, 1), xlabel="confidence (dot = baseline, bar = range over all ±20% perturbations)")
ax.set_title("Sensitivity of each confidence score")
plt.tight_layout()
plt.show()

## 11. Ablation: what does each structural level add?
Hemoglobin α and ubiquitin have all four levels. Removing one level at a time shows how much each contributes,
and that no single level is needed for the right answer (the model degrades gracefully).

In [ ]:
def without(ev, level):
    copy = json.loads(json.dumps(ev))
    copy[level] = {"ran": False}
    return copy

rows = []
for p in EVIDENCE:
    if not all(p[level]["ran"] for level in ("primary", "secondary", "tertiary", "quaternary")):
        continue
    row = {"Protein": p["name"], "all four levels": f"{predict(p)['score']:.2f}"}
    for level in ("primary", "secondary", "tertiary", "quaternary"):
        r = predict(without(p, level))
        row[f"without {level}"] = "undetermined" if r["score"] is None else f"{r['score']:.2f} ({r['specificity']})"
    rows.append(row)
display(pd.DataFrame(rows).set_index("Protein"))

Removing the **primary** level drops both proteins to a fold-level hint: sequence homology carries the specific
function, while the structural levels confirm it. Removing any single structural level lowers the confidence a
little but never changes the prediction.

## 12. Limitations and how they are handled
* **Not calibrated.** The probabilities come from published studies (largely on enzymes) rather than from fitting on
  our own data, so the score ranks evidence transparently but is not a calibrated probability. A calibration would
  need a benchmark of proteins with known function, analysed with self-matches removed (hits ≥ 95% identity
  excluded), comparing predicted confidence with the observed fraction correct.
* **Dependence between levels** is handled by design: only the route provides the base probability, and the other
  levels give small, asymmetric adjustments rather than independent votes.
* **Top hit only.** The primary notebook reports the single best Swiss-Prot hit; for proteins identical across many
  species (e.g. ubiquitin) this may be an orthologue from another organism, which does not change the function.
* **Multidomain proteins.** A Pfam-only prediction names one family; proteins with several domains may combine functions.
* **Quaternary context** is reported for the template; it transfers reliably only for close homologues (≥ 30–40% identity).

## 13. Conclusion
The integration step predicts function by **annotation transfer from the best-supported template** and estimates
confidence as **P(homology) × P(function conserved)**, anchored to published relationships between sequence identity
and functional conservation. The secondary, tertiary and quaternary levels then act as **consistency checks**,
with deliberately asymmetric adjustments because agreement between dependent levels is expected while
contradictions are informative. Every prediction states its **specificity** (protein, family or fold), adds
**biological context** from the quaternary level (oligomeric state, complexes, GO terms), and shows each step of its
reasoning. On the proteins analysed here, no ±20% change of any parameter alters a predicted function, the model
degrades gracefully when levels are missing, and it avoids the over-confidence of a naive weighted average.

## References
1. Tian W, Skolnick J (2003). How well is enzyme function conserved as a function of pairwise sequence identity? *J Mol Biol* 333:863–882.
2. Rost B (1999). Twilight zone of protein sequence alignments. *Protein Eng* 12:85–94.
3. Rost B (2002). Enzyme function less conserved than anticipated. *J Mol Biol* 318:595–608.
4. Aloy P, Ceulemans H, Stark A, Russell RB (2003). The relationship between sequence and interaction divergence in proteins. *J Mol Biol* 332:989–998.
5. van Kempen M, Kim SS, Tumescheit C, et al. (2024). Fast and accurate protein structure search with Foldseek. *Nat Biotechnol* 42:243–246.